# Phần 1: Lý Thuyết Data Fitting và Minh Họa
Notebook này sử dụng dữ liệu giả lập (synthetic data) để kiểm chứng các hàm đã được cài đặt hoàn toàn bằng Pure Python.

In [1]:
import random
import math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
import statsmodels.api as sm

# Import custom implementations
from ols_implementation import ols_fit, hat_matrix, model_metrics, coef_inference, vif, gauss_markov_monte_carlo
from ridge_lasso import ridge_fit
from residual_analysis import residual_plots
from cross_validation import kfold_cv
from matrix_ops import add_intercept

import warnings
warnings.filterwarnings('ignore')

/mnt/shared_data/workspace/application_math/proj_2/part1/residual_analysis.py:52: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  axes[1, 0].set_ylabel('$\sqrt{|Standardized Residuals|}$')


## 1. Tạo Dữ Liệu Giả Lập (Synthetic Data)

In [ ]:
# Cố định random seed để tái lập kết quả
random.seed(42)
np.random.seed(42)

n = 200
p = 3

# Tạo ma trận X (n x p)
X = [[random.gauss(0, 1) for _ in range(p)] for _ in range(n)]

# True parameters (beta0, beta1, beta2, beta3)
true_beta = [2.5, 1.5, -2.0, 0.5]

# Tạo y
y = []
for i in range(n):
    error = random.gauss(0, 1.2)
    yi = true_beta[0] + true_beta[1]*X[i][0] + true_beta[2]*X[i][1] + true_beta[3]*X[i][2] + error
    y.append(yi)
    
X_np = np.array(X)
y_np = np.array(y)

## 2. Kiểm chứng OLS Fit & Suy diễn hệ số

In [ ]:
print("--- CUSTOM PURE PYTHON OLS ---")
beta_hat, sigma2 = ols_fit(X, y)
print("Beta hat:", [round(b, 4) for b in beta_hat])
print("Sigma^2:", round(sigma2, 4))

print("\n--- SKLEARN OLS ---")
reg = LinearRegression().fit(X_np, y_np)
print("Beta hat (sklearn):", [round(reg.intercept_, 4)] + [round(b, 4) for b in reg.coef_])

### Kiểm chứng Coef Inference (Standard errors, p-values)

In [ ]:
print("--- CUSTOM INFERENCE ---")
inference = coef_inference(X, y, beta_hat, sigma2)
print("Standard Errors:", [round(se, 4) for se in inference['se']])
print("P-values:", [round(p, 4) for p in inference['p_value']])

print("\n--- STATSMODELS OLS (Để kiểm chứng SE và p-value) ---")
X_sm = sm.add_constant(X_np)
model_sm = sm.OLS(y_np, X_sm).fit()
print("Standard Errors (SM):", [round(se, 4) for se in model_sm.bse])
print("P-values (SM):", [round(p, 4) for p in model_sm.pvalues])

## 3. Kiểm chứng Hat Matrix và Model Metrics

In [ ]:
print("--- HAT MATRIX ---")
H = hat_matrix(X)
print("Kích thước ma trận H:", len(H), "x", len(H[0]))
print("H[0][0]:", round(H[0][0], 4))

print("\n--- MODEL METRICS ---")
# Dự đoán y_hat
y_hat = []
for i in range(n):
    y_hat_i = beta_hat[0] + sum(beta_hat[j+1]*X[i][j] for j in range(p))
    y_hat.append(y_hat_i)
    
metrics = model_metrics(y, y_hat, p)
print("R-squared:", round(metrics['R_squared'], 4))
print("Adjusted R-squared:", round(metrics['Adj_R_squared'], 4))
print("Sklearn R-squared:", round(reg.score(X_np, y_np), 4))

## 4. Kiểm chứng VIF (Variance Inflation Factor)

In [ ]:
print("--- CUSTOM VIF ---")
vifs = vif(X)
print("VIF cho các biến:", [round(v, 4) for v in vifs])

## 5. Kiểm chứng Ridge Regression & Vẽ Ridge Trace

In [ ]:
lam_values = [0.1, 1.0, 10.0, 100.0, 1000.0]
print("--- CUSTOM RIDGE ---")
ridge_betas = ridge_fit(X, y, lam_values, plot=True)
print("Ridge betas for lambda=10.0:", [round(b, 4) for b in ridge_betas[2]])

print("\n--- SKLEARN RIDGE (lambda=10.0) ---")
ridge_sk = Ridge(alpha=10.0).fit(X_np, y_np)
print("Sklearn Ridge betas:", [round(ridge_sk.intercept_, 4)] + [round(b, 4) for b in ridge_sk.coef_])

## 6. Vẽ Biểu Đồ Phân Tích Phần Dư (Residual Analysis)

In [ ]:
print("--- RESIDUAL PLOTS ---")
cooks_d = residual_plots(X, y, beta_hat)

## 7. Kiểm chứng Cross-Validation

In [ ]:
print("--- CUSTOM K-FOLD CV ---")
cv_score, fold_scores = kfold_cv(X, y, k=5, random_state=42)
print("Mean MSE CV:", round(cv_score, 4))
print("Fold MSEs:", [round(s, 4) for s in fold_scores])

## 8. Minh Họa Định Lý Gauss-Markov bằng Monte Carlo

In [1]:
print("--- GAUSS-MARKOV MONTE CARLO ---")
mc_results = gauss_markov_monte_carlo(n_simulations=500, n=50, p=2, true_beta=[1.0, 2.0, -1.5], sigma=1.0)

print("True Beta:      ", mc_results['true_beta'])
print("Mean Beta OLS:  ", [round(b, 4) for b in mc_results['mean_ols']])
print("Mean Beta Other:", [round(b, 4) for b in mc_results['mean_other']])
print("\nVar Beta OLS:   ", [round(v, 4) for v in mc_results['var_ols']])
print("Var Beta Other: ", [round(v, 4) for v in mc_results['var_other']])
print("\nNhận xét: Giá trị trung bình của cả 2 ước lượng đều xấp xỉ True Beta (Không chệch). Tuy nhiên, phương sai của ước lượng OLS luôn nhỏ hơn phương sai của ước lượng tuyến tính khác, minh họa cho tính chất Tốt Nhất (Best) - BLUE.")

--- GAUSS-MARKOV MONTE CARLO ---


NameError: name 'gauss_markov_monte_carlo' is not defined